### 0. Utils

In [1]:
import osmnx as ox
import networkx as nx
import folium
import random
import geopandas as gpd
import pandas as pd
import io
import rasterio

import sqlite3
import pandas as pd
from sqlalchemy import create_engine
import numpy as np
import matplotlib.pyplot as plt

from scipy.ndimage import zoom, gaussian_filter
from rasterio.merge import merge
from shapely.geometry import Point, LineString
import numpy as np


In [2]:
def get_table_definition(df):
    """
    데이터프레임을 입력받아 컬럼명, 타입, 결측치, 고유값 수, 샘플을 포함한 정의서 반환함.
    """
    definition = pd.DataFrame({
        '컬럼명': df.columns,
        '데이터 타입': df.dtypes.values,
        '비결측치 수': df.count().values,
        '결측치 수': df.isnull().sum().values,
        '고유값 수': df.nunique().values,
        '샘플 데이터': [df[col].iloc[0] if not df[col].empty else None for col in df.columns]
    })
    
    # 결측 비율 추가 (분석 정밀도 향상용)
    definition['결측 비율(%)'] = (definition['결측치 수'] / len(df) * 100).round(2)
    
    return definition

def aggressive_crop(dem_array, transform, pad=2):
    # 기존 crop_valid_data 로직 수행
    mask = ~np.isnan(dem_array)
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    
    # 테두리를 pad(픽셀)만큼 더 안쪽으로 좁힘
    rmin, rmax = rmin + pad, rmax - pad
    cmin, cmax = cmin + pad, cmax - pad
    
    cropped = dem_array[rmin:rmax+1, cmin:cmax+1]
    new_trans = transform * transform.translation(cmin, rmin)
    
    return cropped, new_trans

def safe_gaussian_filter(array, sigma):
    # array에 nan값이 존재하는 경우 정상적 업스케일링이 수행되지 않음
    # 1. NaN마스크 생성 및 0 치환
    nan_mask = np.isnan(array)
    replaced_array = np.where(nan_mask, 0, array)

    # 2. 유효 데이터에 대한 가우시안 필터 적용
    v = gaussian_filter(replaced_array, sigma = sigma)

    # 3. 가중치 보정용 필터 적용(분모)
    w = gaussian_filter(np.logical_not(nan_mask).astype(float), sigma=sigma)

    # 4. 가중치로 나누어 결과 산출 (0으로 나누기 방지)
    result = v / (w + 1e-10)

    return result

def get_elevation_for_line(line, src):
    # 라인의 각 정점 좌표 추출
    coords = list(line.coords)
    # 해당 좌표의 DEM 값 샘플링
    elevations = [val[0] for val in src.sample(coords)]
    return elevations

def to_3d_geometry(row):
    geom = row.geometry
    elevs = row.elevation
    
    # 1. Point 타입 처리 (Nodes 데이터)
    if geom.geom_type == 'Point':
        try:
            # 리스트 형태([z])로 들어온 경우 첫 번째 값 추출
            z = elevs[0] if isinstance(elevs, (list, np.ndarray)) else elevs
            return Point(geom.x, geom.y, z)
        except (IndexError, TypeError):
            return geom

    # 2. LineString 타입 처리 (Edges 데이터)
    elif geom.geom_type == 'LineString':
        try:
            # 기존 (x, y, z) 튜플 결합 로직 유지
            new_coords = [(x, y, z) for (x, y), z in zip(geom.coords, elevs)]
            return LineString(new_coords)
        except Exception:
            return geom
            
    return geom # 기타 타입은 원본 반환

### 1. 테이블 스키마 추출 및 인덱스 설정

#### 1.1. 스키마 추출

In [5]:
conn = sqlite3.connect('seoul_network.db')  # 데이터베이스 파일 경로로 변경

# 1. edges 테이블 구조 확인
edges_schema = pd.read_sql("PRAGMA table_info(edges);", conn)
print("--- Edges Schema ---")
print(edges_schema)

# 2. nodes 테이블 구조 확인
nodes_schema = pd.read_sql("PRAGMA table_info(nodes);", conn)
print("--- Nodes Schema ---")
print(nodes_schema)

--- Edges Schema ---
    cid      name     type  notnull dflt_value  pk
0     0         u   BIGINT        0       None   0
1     1         v   BIGINT        0       None   0
2     2       key   BIGINT        0       None   0
3     3     osmid     TEXT        0       None   0
4     4   highway     TEXT        0       None   0
5     5      name     TEXT        0       None   0
6     6    oneway  BOOLEAN        0       None   0
7     7  reversed     TEXT        0       None   0
8     8    length    FLOAT        0       None   0
9     9  geometry     TEXT        0       None   0
10   10     lanes     TEXT        0       None   0
11   11  maxspeed     TEXT        0       None   0
12   12     width     TEXT        0       None   0
13   13    bridge     TEXT        0       None   0
14   14    tunnel     TEXT        0       None   0
15   15       ref     TEXT        0       None   0
16   16   service     TEXT        0       None   0
17   17  junction     TEXT        0       None   0
18   18   

#### 1.2. pk 설정

In [6]:
# DB 연결
conn = sqlite3.connect("seoul_network.db")
cur = conn.cursor()

# 1. 노드 테이블 유니크 인덱스 생성
cur.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_nodes_osmid ON nodes(osmid);")

# 2. 엣지 테이블 연결성 인덱스 생성
cur.execute("CREATE INDEX IF NOT EXISTS idx_edges_source_target ON edges(u, v);")

# 변경사항 저장 및 종료
conn.commit()
conn.close()

# pk는 아니라도 유니크한 인덱스를 생성하여 검색 성능을 향상

In [7]:
# DB 연결
conn = sqlite3.connect("seoul_network.db")

query = """SELECT * 
           FROM edges
           LIMIT 3
           ;"""  # 예시 osmid 값

df_top3 = pd.read_sql(query, conn)

### 2. 고도계 파일 생성 (1회)

기존에는 nodes에 elevation을 매핑하는 것을 고려하였으나, edges 데이터에도 WGS84 좌표가 linestring object로 존재한다는 점을 고려하여 edges에 elevation을 매핑함 
브이월드에 WGS84로 발표되는 고해상도 고도데이터가 있는지 확인

공개정보에서 30m급 해상도 DEM 데이터 

#### 2.1. 읽기 및 업스케일링

In [3]:
# DEM 데이터 읽기
DEM_list = ["37608_seoul.img", "37612_anyang.img", "37705_seongdong.img", "37709_suwon.img"]
result_dict = {}

In [4]:
# 1. 파일 핸들 리스트 생성 (Context Manager 유지)
src_files_to_mosaic = []
for dem_file in DEM_list:
    src = rasterio.open(dem_file)
    src_files_to_mosaic.append(src)

# 모든 파일의 CRS가 0번 파일과 동일한지 검증
for i, src in enumerate(src_files_to_mosaic):
    if src.crs != src_files_to_mosaic[0].crs:
        raise ValueError(f"파일 {i}의 CRS가 일치하지 않음: {src.crs}")

In [5]:
# 2. Merge 수행 (지리적 위치 기반 자동 병합, 별도 순서 지정 필요 X)
# mosaic: 병합된 numpy array
# out_trans: 병합된 이미지의 새로운 Affine Transform
mosaic, out_trans = merge(src_files_to_mosaic)
mosaic_float = mosaic[0].astype('float32')
mosaic_float[mosaic_float == src_files_to_mosaic[0].nodata] = np.nan
final_dem, new_trans = aggressive_crop(mosaic_float, out_trans, pad = 3)

zoom_dem = zoom(final_dem, zoom= 4, order = 2)
smooth_dem = safe_gaussian_filter(zoom_dem, 1)

#### 2.2. 저장하기

In [24]:
# 1. 확대한 배율(zoom_factor)만큼 해상도 조정
# new_trans는 aggressive_crop 결과물이므로 이를 기준으로 스케일 조정
zoom_factor = 4
final_transform = new_trans * new_trans.scale(
    (1 / zoom_factor), 
    (1 / zoom_factor)
)

# 2. 파일 저장 파라미터 설정
# smooth_dem의 shape을 기준으로 width, height 할당
save_meta = {
    'driver': 'GTiff',
    'dtype': 'float32',
    'nodata': np.nan,
    'width': smooth_dem.shape[1],
    'height': smooth_dem.shape[0],
    'count': 1,
    'crs': src_files_to_mosaic[0].crs, # 원본 CRS 계승 (EPSG:5186)
    'transform': final_transform
}

# 3. GeoTIFF 파일 생성 및 쓰기
with rasterio.open('mosaic_dem.tif', 'w', **save_meta) as dst:
    dst.write(smooth_dem, 1)

# 4. 사용한 리소스 해제
for src in src_files_to_mosaic:
    src.close()

### 3. 고도 할당

#### 3.1. OSM 데이터 다운로드 (1회)

In [11]:
# 1. 특정 지역의 도로망 데이터 가져오기 
place_name = "Seoul, South Korea"

# all = 모든 경로 / all_private = 사유지 도로 포함 경로 / drive = 자동차주행경로
# walk = 보행가능경로 / bike = 자전거주행경로

try:
    G = ox.graph_from_place(place_name, network_type="all")
    print(f"다운로드 완료: 노드 {len(G.nodes)}개, 엣지(도로) {len(G.edges)}개")
except Exception as e:
    print(f"데이터 다운로드 실패. 다른 지역명을 시도하거나 네트워크 상태를 확인하세요: {e}")
    exit()

다운로드 완료: 노드 169847개, 엣지(도로) 466335개


In [ ]:
# 2. 도로망 데이터를 GeoDataFrame으로 변환
nodes, edges = ox.graph_to_gdfs(G)

edges_path = "seoul_edges.shp"

# 2. 리스트/딕셔너리 타입 컬럼 처리 (Shapefile 제약 사항)
# OSM 데이터는 한 컬럼에 여러 값이 리스트로 들어있는 경우가 많아 변환 필수
for col in edges.columns:
    if isinstance(edges[col].iloc[0], list):
        edges[col] = edges[col].astype(str)

# 3. 파일 저장
edges.to_file(edges_path, encoding='cp949')

/tmp/ipykernel_14445/3676876313.py:15: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  nodes.to_file(nodes_path, encoding='cp949')
/home/joon/miniconda3/envs/sea-env/lib/python3.10/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'street_count' to 'street_cou'
  ogr_write(


#### 3.2. OSM 데이터에 DEM 고도 정보 추가하기

In [3]:
# 도로 데이터 로드 및 변환
roads = gpd.read_file('seoul_edges.shp')
nodes = gpd.read_file('seoul_nodes.shp')
# 좌표계 변환 
roads = roads.to_crs("EPSG:5179")
nodes = nodes.to_crs("EPSG:5179")

In [4]:
# 결과 저장을 위한 빈 리스트
road_elevation_data = []
node_elevation_data = []

with rasterio.open('mosaic_dem.tif') as src:
    # 도로 데이터의 각 행을 명시적으로 순회
    print("roads 고도값 매핑 시작")
    for geom in roads.geometry:
        try:
            # 1. 정점 좌표 추출
            coords = list(geom.coords)
            # print(f"처리 중인 도로의 첫 번째 정점 좌표: {coords[0]}")  # 디버깅용 출력
            # 2. Raster 샘플링 (모든 정점에 대한 고도값 추출)
            # src.sample은 제너레이터를 반환하므로 리스트로 변환
            sample_gen = src.sample(coords)
            elevs = [val[0] for val in sample_gen]
            
            road_elevation_data.append(elevs)
            # print(f"추출된 고도값: {elevs}")  # 디버깅용 출력
            
        except Exception as e:
            # 예외 발생 시(범위 외 좌표 등) NaN 처리
            road_elevation_data.append([np.nan])
            print(f"도로 처리 중 오류 발생: {e}")  # 오류 로그 출력
    print("nodes 고도값 매핑 시작")
    for geom in nodes.geometry:
        try:
            coords = list(geom.coords)
            sample_gen = src.sample(coords)
            elevs = [val[0] for val in sample_gen]
            node_elevation_data.append(elevs)
        except Exception as e:
            node_elevation_data.append([np.nan])
            print(f"노드 처리 중 오류 발생: {e}")

# 별도의 elevation 컬럼에 결과 할당 (geometry 오염 방지)
roads['elevation'] = road_elevation_data  # 도로 데이터에 해당하는 고도값만 할당
nodes['elevation'] = node_elevation_data  # 노드 데이터에 해당하는 고도값만 할당

# axis=1 설정을 통해 각 '행(Row)'을 함수로 전달함
roads['geometry'] = roads.apply(to_3d_geometry, axis=1)

roads 고도값 매핑 시작
nodes 고도값 매핑 시작


In [5]:
nodes['geometry'] = nodes.apply(to_3d_geometry, axis=1)

In [6]:
roads.drop(columns=['elevation'], inplace=True)  # 중간 컬럼 제거 
nodes.drop(columns=['elevation'], inplace=True)  # 중간 컬럼 제거 

#### 3.3. 고도데이터 포함된 데이터 저장

In [10]:
nodes

,osmid,y,x,street_cou,junction,highway,ref,railway,geometry
0,278159482,37.525964,126.997422,3,None,None,None,None,POINT Z (955592.375 1947525.468 42.084)
1,282723724,37.588055,127.023541,3,None,None,None,None,POINT Z (957935.123 1954402.092 34.748)
2,282723780,37.586624,127.025199,3,None,None,None,None,POINT Z (958080.754 1954242.599 43.894)
3,282723804,37.586564,127.025316,3,None,None,None,None,POINT Z (958090.997 1954235.935 43.327)
4,282723942,37.586235,127.025275,3,None,None,None,None,POINT Z (958087.193 1954199.408 43.85)
...,...,...,...,...,...,...,...,...,...
169842,13793563322,37.560727,126.935929,3,None,None,None,None,POINT Z (950181.954 1951413.008 29.198)
169843,13793646181,37.562888,126.936877,4,None,None,None,None,POINT Z (950267.056 1951652.337 35.435)
169844,13793646184,37.562878,126.937757,3,None,None,None,None,POINT Z (950344.831 1951650.762 33.413)
169845,13793646186,37.562779,126.937599,3,None,None,None,None,POINT Z (950330.784 1951639.84 32.778)


In [7]:
roads.to_file("roads_final.gpkg", driver="GPKG")
nodes.to_file("nodes_final.gpkg", driver="GPKG")

In [8]:
print("save completed")

save completed


### 99. TEST ; 시각화